# Reading FITS files

This notebook is used to explore the FITS files located in the `../data` directory.

In [ ]:
from astropy.io import fits
import glob
import os
import pandas as pd
import sys
import getpass
from multiprocessing import BoundedSemaphore, Lock

In [ ]:
# Add hsc_to_lsst to python path to import pipeline
sys.path.insert(0, os.path.abspath("HSC_to_LSST"))
try:
    from hsc_to_lsst.pipeline import query_and_write
    print("Successfully imported query_and_write from hsc_to_lsst.pipeline")
except ImportError as e:
    print(f"Error importing hsc_to_lsst: {e}")
    print("Make sure the HSC_to_LSST directory is in the current directory or python path.")

In [ ]:
# List fits files in one of final-sample images directory (e.g. udeep)
data_dir = "../data/final-sample/simct_images/udeep"
fits_files = glob.glob(os.path.join(data_dir, "*.fits"))
print(f"Found {len(fits_files)} FITS files in {data_dir}:")
# Print first 5 files for brevity
for f in fits_files[:5]:
    print(f)


In [ ]:
def explore_fits(file_path):
    print(f"\n{'='*80}\nExploring: {file_path}\n{'='*80}")
    try:
        with fits.open(file_path) as hdul:
            hdul.info()
            
            for i, hdu in enumerate(hdul):
                print(f"\n--- HDU {i} ({hdu.name}) ---")
                display(hdu.header)
                
                if isinstance(hdu, fits.BinTableHDU) or isinstance(hdu, fits.TableHDU):
                    print(f"Table columns: {hdu.columns.names}")
                    # Convert to pandas data frame for nicer display of first few rows
                    try:
                        from astropy.table import Table
                        df = Table(hdu.data).to_pandas()
                        display(df.head())
                        print(f"Table shape: {df.shape}")
                    except Exception as e:
                        print(f"Could not convert to DataFrame: {e}")
                elif isinstance(hdu, fits.ImageHDU) or isinstance(hdu, fits.PrimaryHDU):
                     if hdu.data is not None:
                        print(f"Image data shape: {hdu.data.shape}")
                        print(f"Min: {hdu.data.min()}, Max: {hdu.data.max()}")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

In [ ]:
# Load final-sample dataset
import os
import pandas as pd

samples = ['hectomap', 'udeep', 'w03_gama09h', 'w05_vvds', 'w07_aegis', 'xmm']
data_dir = "../data/final-sample"

# Coordinate string converter helper
def _coord_str(ra, dec):
    ra_h = ra / 15.0
    h = int(ra_h); m = int((ra_h - h) * 60); s = (ra_h - h - m / 60) * 3600
    sign = "+" if dec >= 0 else "-"
    adec = abs(dec)
    dh = int(adec); dm = int((adec - dh) * 60); ds = (adec - dh - dm / 60) * 3600
    return f"{h:02d}{m:02d}{s:06.3f}{sign}{dh:02d}{dm:02d}{ds:06.3f}"

all_dfs = []
for s in samples:
    csv_path = os.path.join(data_dir, "params", f"{s}_finalpar.csv")
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path, dtype={"galaxy_id": str})
            df["sample_name"] = s
            # Create aliases for downstream compatibility
            df["object_id"] = df["galaxy_id"]
            df["ra"] = df["galaxy_ra"]
            df["dec"] = df["galaxy_dec"]
            all_dfs.append(df)
            print(f"Loaded sample '{s}' with {len(df)} rows.")
        except Exception as e:
            print(f"Error loading {csv_path}: {e}")
    else:
        print(f"Warning: {csv_path} not found.")

combined_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
print(f"Total combined catalog shape: {combined_df.shape}")

# Set deep to combined_df to process all galaxies
deep = combined_df
wide = None

if deep is not None and not deep.empty:
    print(f"\nSet 'deep' to complete catalog with shape: {deep.shape}")
    print("\n--- Combined Catalog DataFrame (First 5 Rows) ---")
    display(deep.head())


## Query HSC Database

The following cells allow you to query the HSC database for the objects in the DataFrames. You will need to provide your HSC SSP username and password.

In [ ]:
# No HSC credentials needed for local Margherita data
hsc_username = None
hsc_password = None


In [ ]:
# Skipping HSC database queries as we are using the local FITS files from the Margherita sample directly in the following cells.
print("Skipping HSC database queries: using local Margherita FITS files directly.")


## Display Retrieved Images

The following cell displays the images retrieved from the HSC database.

In [ ]:
# Note: Simulated images are 99x99 pixels at a pixel scale of 0.168 arcsec/pixel.
import os
from astropy.io import fits
try:
    import ipywidgets
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

def _coord_str(ra, dec):
    ra_h = ra / 15.0
    h = int(ra_h); m = int((ra_h - h) * 60); s = (ra_h - h - m / 60) * 3600
    sign = "+" if dec >= 0 else "-"
    adec = abs(dec)
    dh = int(adec); dm = int((adec - dh) * 60); ds = (adec - dh - dm / 60) * 3600
    return f"{h:02d}{m:02d}{s:06.3f}{sign}{dh:02d}{dm:02d}{ds:06.3f}"

def verify_and_read_fits(df, base_dir, limit=None):
    print("Reading and verifying FITS files...")
    count = 0
    missing = 0
    
    # Select subset if limit is specified
    iterator = df.itertuples()
    total_len = len(df)
    if limit is not None:
        iterator = list(iterator)[:limit]
        total_len = limit
        
    for row in tqdm(iterator, total=total_len, desc="Reading FITS"):
        coord = _coord_str(row.ra, row.dec)
        sim_path = os.path.join(base_dir, "simct_images", row.sample_name, f"imoutf_{coord}.fits")
        
        if not os.path.exists(sim_path):
            missing += 1
            continue
            
        try:
            with fits.open(sim_path) as hdul:
                # Just read data shape to verify validity
                _ = hdul[0].data.shape
            count += 1
        except Exception as e:
            print(f"Error reading FITS for galaxy {row.object_id}: {e}")
            
    print(f"Finished reading FITS: {count} valid files read, {missing} files missing on disk.")

# Verify images for final-sample Deep catalog
final_sample_dir = "../data/final-sample"
if deep is not None:
    print("\n--- Verifying Final Sample Images ---")
    verify_and_read_fits(deep, final_sample_dir, limit=None)


## Apply LSST Degradation

This section applies the `read_degrade_hsc` function to the retrieved HSC images to simulate LSST-like images.
It then displays the original HSC images side-by-side with the degraded LSST images.

### Note on Image Sizes and PSF

- **Image Sizes**: The HSC images are approximately 20 arcseconds on a side (119x119 pixels at 0.168"/pix). The LSST images are 8.2 arcseconds on a side (41x41 pixels at 0.2"/pix). In the comparison plots below, the **HSC images are zoomed in** to match the field of view of the LSST images (8.2" x 8.2") to allow for a direct comparison of features at the same angular size.
- **PSF Model**: The LSST PSF is modeled as a Gaussian kernel. The FWHM (Full Width at Half Maximum) of this Gaussian is sampled from LSST DP0.2 simulations, which provide realistic seeing conditions expected for the LSST survey.

In [ ]:
from hsc_to_lsst.pipeline import degrade_images
from hsc_to_lsst.lsst_props import dp0_gmm_sampler
from multiprocessing import Lock
import os
import numpy as np
from astropy.io import fits
try:
    import ipywidgets
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# Note: Simulated images are 99x99 pixels at a pixel scale of 0.168 arcsec/pixel.
def _coord_str(ra, dec):
    ra_h = ra / 15.0
    h = int(ra_h); m = int((ra_h - h) * 60); s = (ra_h - h - m / 60) * 3600
    sign = "+" if dec >= 0 else "-"
    adec = abs(dec)
    dh = int(adec); dm = int((adec - dh) * 60); ds = (adec - dh - dm / 60) * 3600
    return f"{h:02d}{m:02d}{s:06.3f}{sign}{dh:02d}{dm:02d}{ds:06.3f}"

def perform_degradation_margherita(df, base_dir, output_dir, coadd_years=1, limit=None):
    print(f"Degrading images to {output_dir}...")
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize sampler
    dp0_sampler = dp0_gmm_sampler(str(coadd_years))
    log_lock = Lock()
    log_file = os.path.join(output_dir, "degrade.log")
    
    count = 0
    # Select subset if limit is specified
    iterator = df.itertuples()
    total_len = len(df)
    if limit is not None:
        iterator = list(iterator)[:limit]
        total_len = limit
        
    for row in tqdm(iterator, total=total_len, desc="Degrading"):
        coord = _coord_str(row.ra, row.dec)
        sim_path = os.path.join(base_dir, "simct_images", row.sample_name, f"imoutf_{coord}.fits")
        
        if not os.path.exists(sim_path):
            continue
            
        sample_out_dir = os.path.join(output_dir, "simct_images", row.sample_name)
        out_path = os.path.join(sample_out_dir, f"imoutf_{coord}.fits")
        if os.path.exists(out_path):
            count += 1
            continue
            
        try:
            with fits.open(sim_path) as hdul:
                cube = hdul[0].data.astype(np.float32)  # shape (5, H, W)
                
            images = [cube[i] for i in range(5)]
            # Use None for PSFs, so default mean FWHMs are used
            psfs = [None] * 5
            
            success, degraded_images, mag_change = degrade_images(
                images,
                psfs,
                dp0_sampler,
                hsc_pix_scale=0.168,
                zp_rms_frac_thresh=0.3,
                zp_max_mag_change=2.0,
                lsst_size_pix=41,
                bands='grizy',
                log_file=log_file,
                log_lock=log_lock,
                log_prefix=f"{row.object_id}: "
            )
            
            if success:
                # Save degraded images as a single 3D FITS cube (5, 41, 41) matching input filename
                sample_out_dir = os.path.join(output_dir, "simct_images", row.sample_name)
                os.makedirs(sample_out_dir, exist_ok=True)
                
                out_path = os.path.join(sample_out_dir, f"imoutf_{coord}.fits")
                degraded_cube = np.stack(degraded_images, axis=0) # shape (5, 41, 41)
                
                hdu = fits.PrimaryHDU(data=degraded_cube)
                hdu.writeto(out_path, overwrite=True)
        except Exception as e:
            err_msg = f'Error processing: {e}'
            print(err_msg)
            try:
                with log_lock:
                    with open(log_file, "a") as lf:
                        lf.write(f'{err_msg}\n')
            except Exception:
                pass
            
        count += 1
    print(f"Degradation completed for {count} galaxies.")

# Run degradation and display for final-sample Deep Data
final_sample_dir = "../data/final-sample"
degraded_dir = "../data/final-sample-output"
if deep is not None:
    print("\n--- Degrading Final Sample Data ---")
    perform_degradation_margherita(deep, final_sample_dir, degraded_dir, limit=None)
